#SETUP

In [2]:
%%capture --no-stderr
%pip install -U langgraph


In [3]:
import importlib.metadata
print(importlib.metadata.version("langgraph"))


1.0.3


In [5]:
from typing_extensions 
import TypedDict, NotRequired
class State(TypedDict):
    user_name: str
    graph_state: str
    mood: NotRequired[str]
    previous_graph_state: NotRequired[str]


In [6]:
import difflib
from datetime import datetime

def make_diff(before: str, after: str) -> str:
    before_lines = (before or "").splitlines()
    after_lines = (after or "").splitlines()
    return "\n".join(difflib.unified_diff(before_lines, after_lines, lineterm=""))

def node_1(state: dict) -> dict:
    user = state.get("user_name") or "User"
    prev = state.get("graph_state") or ""
    print(f"--- Node 1 for {user} ---")
    new_text = prev + " I am"
    diff = make_diff(prev, new_text)
    return {
        "graph_state": new_text,
        "diff": diff,
        "last_updated": datetime.utcnow().isoformat(),
        "mood": "neutral",
        "previous_graph_state": prev,
    }

def node_2(state: dict) -> dict:
    user = state.get("user_name") or "User"
    prev = state.get("graph_state") or ""
    print(f"--- Node 2 (happy) for {user} ---")
    new_text = prev + " happy!"
    diff = make_diff(prev, new_text)
    return {
        "graph_state": new_text,
        "diff": diff,
        "last_updated": datetime.utcnow().isoformat(),
        "mood": "happy",
        "previous_graph_state": prev,
    }

def node_3(state: dict) -> dict:
    user = state.get("user_name") or "User"
    prev = state.get("graph_state") or ""
    print(f"--- Node 3 (sad) for {user} ---")
    new_text = prev + " sad!"
    diff = make_diff(prev, new_text)
    return {
        "graph_state": new_text,
        "diff": diff,
        "last_updated": datetime.utcnow().isoformat(),
        "mood": "sad",
        "previous_graph_state": prev,
    }

if __name__ == "__main__":
    state = {"user_name": "Akshita", "graph_state": ""}
    state.update(node_1(state))
    state.update(node_2(state))
    state.update(node_3(state))
    print("Final state:", state["graph_state"])
    print("Last diff:\n", state["diff"])


--- Node 1 for Akshita ---
--- Node 2 (happy) for Akshita ---
--- Node 3 (sad) for Akshita ---
Final state:  I am happy! sad!
Last diff:
 --- 
+++ 
@@ -1 +1 @@
- I am happy!
+ I am happy! sad!


C:\Users\Admin\AppData\Local\Temp\ipykernel_35340\460188994.py:18: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "last_updated": datetime.utcnow().isoformat(),
C:\Users\Admin\AppData\Local\Temp\ipykernel_35340\460188994.py:32: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "last_updated": datetime.utcnow().isoformat(),
C:\Users\Admin\AppData\Local\Temp\ipykernel_35340\460188994.py:46: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "last_updated": datetime.utcnow().isoformat(),


In [7]:
import random
from typing import Literal

def choose_mood(state) -> Literal["node_2", "node_3"]:
    text = (state.get("graph_state") or "").lower()
    if any(x in text for x in ["sad", "unhappy", "down", "not good"]):
        return "node_3"
    return "node_2" if random.random() < 0.6 else "node_3"

if __name__ == "__main__":
    state = {"graph_state": "I am feeling a bit down today"}
    result = choose_mood(state)
    print("Next node:", result)

    state2 = {"graph_state": "I am excited and energetic"}
    result2 = choose_mood(state2)
    print("Next node:", result2)


Next node: node_3
Next node: node_2


In [8]:
def merge_state(a, b):
    s = dict(a) if a else {}
    s.update(b or {})
    return s

class Runner:
    def invoke(self, initial, max_steps=3):
        nodes = {"node_1": node_1, "node_2": node_2, "node_3": node_3}
        state = dict(initial) if initial else {}
        print("Starting for", state.get("user_name", "User"))
        results = []
        out = node_1(state)
        results.append(out)
        state = merge_state(state, out)
        for _ in range(max_steps - 1):
            nxt = choose_mood(state)
            print("Next:", nxt)
            fn = nodes.get(nxt)
            if not fn:
                break
            out = fn(state)
            results.append(out)
            state = merge_state(state, out)
        return results

graph = Runner()

if __name__ == "__main__":
    initial = {"user_name": "Akshita", "graph_state": ""}
    output = graph.invoke(initial)
    print("Runs:", len(output))
    print("Final:", output[-1]["graph_state"])


Starting for Akshita
--- Node 1 for Akshita ---
Next: node_2
--- Node 2 (happy) for Akshita ---
Next: node_3
--- Node 3 (sad) for Akshita ---
Runs: 3
Final:  I am happy! sad!


C:\Users\Admin\AppData\Local\Temp\ipykernel_35340\460188994.py:18: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "last_updated": datetime.utcnow().isoformat(),
C:\Users\Admin\AppData\Local\Temp\ipykernel_35340\460188994.py:32: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "last_updated": datetime.utcnow().isoformat(),
C:\Users\Admin\AppData\Local\Temp\ipykernel_35340\460188994.py:46: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "last_updated": datetime.utcnow().isoformat(),


In [9]:
graph.invoke({
    "graph_state": "Hi, this is Akshita.",
    "user_name": "Akshita",
    "previous_graph_state": ""
})


Starting for Akshita
--- Node 1 for Akshita ---
Next: node_3
--- Node 3 (sad) for Akshita ---
Next: node_3
--- Node 3 (sad) for Akshita ---


C:\Users\Admin\AppData\Local\Temp\ipykernel_35340\460188994.py:18: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "last_updated": datetime.utcnow().isoformat(),
C:\Users\Admin\AppData\Local\Temp\ipykernel_35340\460188994.py:46: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "last_updated": datetime.utcnow().isoformat(),


[{'graph_state': 'Hi, this is Akshita. I am',
  'diff': '--- \n+++ \n@@ -1 +1 @@\n-Hi, this is Akshita.\n+Hi, this is Akshita. I am',
  'last_updated': '2025-11-25T08:10:32.987034',
  'mood': 'neutral',
  'previous_graph_state': 'Hi, this is Akshita.'},
 {'graph_state': 'Hi, this is Akshita. I am sad!',
  'diff': '--- \n+++ \n@@ -1 +1 @@\n-Hi, this is Akshita. I am\n+Hi, this is Akshita. I am sad!',
  'last_updated': '2025-11-25T08:10:32.987448',
  'mood': 'sad',
  'previous_graph_state': 'Hi, this is Akshita. I am'},
 {'graph_state': 'Hi, this is Akshita. I am sad! sad!',
  'diff': '--- \n+++ \n@@ -1 +1 @@\n-Hi, this is Akshita. I am sad!\n+Hi, this is Akshita. I am sad! sad!',
  'last_updated': '2025-11-25T08:10:32.987709',
  'mood': 'sad',
  'previous_graph_state': 'Hi, this is Akshita. I am sad!'}]